# Notebook 02 — Exploratory Business Analysis

Business-facing EDA covering attendance trends, revenue analysis, promotion lift, ticket segments, concessions, and merchandise.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.config import DATA_PROCESSED_DIR, DATA_SIMULATED_DIR
sns.set_theme(style='whitegrid')

def load(name):
    for d in [DATA_PROCESSED_DIR, DATA_SIMULATED_DIR]:
        p = d / f'{name}.csv'
        if p.exists(): return pd.read_csv(p)
    return pd.DataFrame()

games = load('games'); games['game_date'] = pd.to_datetime(games['game_date'])
attendance = load('attendance')
ticket_sales = load('ticket_sales')
concessions = load('concessions')
merchandise = load('merchandise')
promotions = load('promotions')
fan_profiles = load('fan_profiles')
print('Data loaded.')

## 1. Attendance Trends

In [ ]:
df = games.merge(attendance, on='game_id').merge(promotions, on='game_id')

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Weekend vs Weekday
df.groupby('is_weekend')['capacity_pct'].mean().plot(kind='bar', ax=axes[0,0], color=['#ff9999','#06d6a0'])
axes[0,0].set_title('Avg Capacity % — Weekend vs Weekday')
axes[0,0].set_xticklabels(['Weekday','Weekend'], rotation=0)

# Rivalry games
df.groupby('rivalry_flag')['capacity_pct'].mean().plot(kind='bar', ax=axes[0,1], color=['#b0bec5','#0068c9'])
axes[0,1].set_title('Avg Capacity % — Rivalry vs Regular')
axes[0,1].set_xticklabels(['Regular','Rivalry'], rotation=0)

# Monthly attendance
month_att = df.groupby('month')['actual_attendance'].mean()
month_att.plot(kind='line', marker='o', ax=axes[1,0], color='steelblue')
axes[1,0].set_title('Avg Attendance by Month')
axes[1,0].set_xlabel('Month')

# Attendance tier distribution
tier_counts = attendance['attendance_tier'].value_counts()
tier_counts.plot(kind='bar', ax=axes[1,1], color=['#ff9999','#ffd166','#06d6a0','#0068c9'])
axes[1,1].set_title('Attendance Tier Distribution')
axes[1,1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

## 2. Revenue Trends

In [ ]:
ticket_agg = ticket_sales.groupby('game_id')['net_ticket_revenue'].sum().reset_index()
con_agg = concessions.groupby('game_id')['gross_revenue'].sum().reset_index().rename(columns={'gross_revenue':'con_rev'})
merch_agg = merchandise.groupby('game_id')['gross_revenue'].sum().reset_index().rename(columns={'gross_revenue':'merch_rev'})

rev = df.merge(ticket_agg, on='game_id').merge(con_agg, on='game_id').merge(merch_agg, on='game_id')
rev['total_revenue'] = rev['net_ticket_revenue'] + rev['con_rev'] + rev['merch_rev']

monthly_rev = rev.groupby('month')[['net_ticket_revenue','con_rev','merch_rev']].mean()
monthly_rev.plot(kind='bar', figsize=(12, 5), color=['#0068c9','#ff6b6b','#ffd166'])
plt.title('Average Monthly Revenue by Category')
plt.xlabel('Month')
plt.ylabel('Avg Revenue ($)')
plt.legend(['Tickets','Concessions','Merchandise'])
plt.tight_layout()
plt.show()

print('Revenue share:')
shares = rev[['net_ticket_revenue','con_rev','merch_rev']].sum()
for col, val in shares.items():
    print(f'  {col}: ${val:,.0f} ({val/shares.sum()*100:.1f}%)')

## 3. Promotion Analysis

In [ ]:
promo_att = df.groupby('promotion_type')['capacity_pct'].mean().sort_values(ascending=False)
promo_att.plot(kind='bar', figsize=(12, 5), color='steelblue')
plt.title('Avg Capacity % by Promotion Type')
plt.xlabel('Promotion Type')
plt.ylabel('Avg Capacity %')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.show()

## 4. Ticket Segment Analysis

In [ ]:
seg_summary = ticket_sales.groupby('segment_name').agg(
    total_net_rev=('net_ticket_revenue','sum'),
    avg_price=('avg_ticket_price','mean'),
    avg_sell_through=('sell_through_rate','mean'),
).sort_values('total_net_rev', ascending=False)
display(seg_summary.round(2))

## 5. Concessions & Merchandise Observations

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

con_cat = concessions.groupby('category')['gross_revenue'].sum().sort_values(ascending=False)
con_cat.plot(kind='bar', ax=axes[0], color='#ff6b6b')
axes[0].set_title('Total Concession Revenue by Category')
axes[0].tick_params(axis='x', rotation=35)

merch_cat = merchandise.groupby('category')['gross_revenue'].sum().sort_values(ascending=False)
merch_cat.plot(kind='bar', ax=axes[1], color='#ffd166')
axes[1].set_title('Total Merchandise Revenue by Category')
axes[1].tick_params(axis='x', rotation=35)

plt.tight_layout()
plt.show()

## 6. Key Business Observations

In [ ]:
weekend_lift = df.groupby('is_weekend')['capacity_pct'].mean().diff().iloc[-1]
rivalry_lift = df.groupby('rivalry_flag')['capacity_pct'].mean().diff().iloc[-1]
print(f'Weekend attendance lift: +{weekend_lift:.1f} percentage points')
print(f'Rivalry game attendance lift: +{rivalry_lift:.1f} percentage points')
top_opp = df.groupby('away_team_name')['capacity_pct'].mean().idxmax()
print(f'Strongest drawing opponent: {top_opp}')
best_promo = promo_att.index[0]
print(f'Best promotion for attendance: {best_promo}')